In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn pyarrow statsmodels

In [ ]:
#Setup
Loading the libraries we need. Pandas for data handling, sklearn for the models, matplotlib and seaborn for plots.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve, ConfusionMatrixDisplay)
from statsmodels.stats.contingency_tables import mcnemar

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 50)

In [ ]:
Loading the dataset

In [ ]:
df = pd.read_parquet('data/qld_health_analysis.parquet')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print('Geography levels:')
print(df['geog_type_name'].value_counts())
print()
print('Conditions:')
for c in df['long_term_health_condition'].unique():
    print(' -', c)
print()
print('Ages:', sorted(df['age_group'].unique()))
print('Sex:', df['sex_name'].unique().tolist())
print('Year:', df['year'].unique().tolist())
print('State:', df['state_name'].unique().tolist())

In [ ]:
#Checking for missing values

In [ ]:
nulls = df.isna().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print('Columns with missing values:')
print(nulls)

In [ ]:
sa2_only = df[df['geog_type_name'] == 'Statistical Area Level 2']
sa2_nulls = sa2_only.isna().sum()
sa2_nulls = sa2_nulls[sa2_nulls > 0]
print('Nulls remaining in SA2-only subset:')
print(sa2_nulls if len(sa2_nulls) else 'None')

In [ ]:
#Filtering and reshaping

In [ ]:
sa2 = df[df['geog_type_name'] == 'Statistical Area Level 2'].copy()
sa2 = sa2[(sa2['age_group'] != '_T') & (sa2['sex_name'] != 'Persons')]
sa2 = sa2.drop(columns=['year', 'state', 'state_name', 'geog_type', 'geog_type_name',
                        'sex', 'lthc', 'age_group_label'])
print('After filtering:', sa2.shape)

In [ ]:
index_cols = ['sa2_code', 'sa2_name', 'sa3_code', 'sa3_name',
              'sa4_code', 'sa4_name', 'PHN_NAME_2023',
              'geog_id', 'age_group', 'sex_name']

wide = sa2.pivot_table(index=index_cols,
                       columns='long_term_health_condition',
                       values='persons',
                       aggfunc='sum').reset_index()
wide.columns.name = None
print('Wide shape:', wide.shape)
wide.head()

In [ ]:
#Feature engineering

In [ ]:
TOTAL = 'Total (Persons)'
MENTAL = 'Mental health condition (including depression or anxiety)'

before = len(wide)
wide = wide[wide[TOTAL] > 0].copy()
print(f'Dropped {before - len(wide)} zero-population cells. Now have {len(wide)}.')

# physical conditions (everything except total, mental, not stated, no condition)
physical = [c for c in wide.columns if c not in index_cols
            and c not in {TOTAL, MENTAL, 'Not stated', 'No long-term health condition(s)'}]

# convert counts to rates
for c in physical + [MENTAL]:
    wide[f'rate__{c}'] = wide[c] / wide[TOTAL]

rate_phys = [f'rate__{c}' for c in physical]

In [ ]:
wide['comorbidity_load'] = wide[rate_phys].sum(axis=1)
wide['comorbidity_count_high'] = (wide[rate_phys] > wide[rate_phys].median()).sum(axis=1)
wide['log_population'] = np.log1p(wide[TOTAL])

age_mid = {'0_14': 7, '15_24': 19.5, '25_34': 29.5, '35_44': 39.5,
           '45_54': 49.5, '55_64': 59.5, '65_74': 69.5, '75_84': 79.5, 'GE85': 90}
wide['age_midpoint'] = wide['age_group'].map(age_mid)

wide[['comorbidity_load', 'comorbidity_count_high', 'log_population', 'age_midpoint']].describe()

In [ ]:
sa4_mean = (wide.groupby('sa4_code')[rate_phys].mean()
                .add_prefix('sa4_mean_').reset_index())
sa4_pop = (wide.groupby('sa4_code')[TOTAL].sum()
              .reset_index().rename(columns={TOTAL: 'sa4_total_pop'}))
sa4_pop['sa4_log_population'] = np.log1p(sa4_pop['sa4_total_pop'])
sa4_pop = sa4_pop.drop(columns=['sa4_total_pop'])

wide = wide.merge(sa4_mean, on='sa4_code').merge(sa4_pop, on='sa4_code')

sa4_cols = [c for c in wide.columns if c.startswith('sa4_mean_') or c == 'sa4_log_population']
print(f'Added {len(sa4_cols)} SA4 regional features.')

In [ ]:
mh_rate = wide[f'rate__{MENTAL}']
threshold = mh_rate.median()
wide['target_high_mh'] = (mh_rate > threshold).astype(int)

print(f'QLD median MH rate (threshold): {threshold:.4f}')
print(wide['target_high_mh'].value_counts(normalize=True).round(3))